In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Dibimbing Data-Engineering Bootcamp 16/raw_clinic_source_data_quality.csv')
df.head()



,id_kunjungan,nik,nama_pasien,email_pasien,no_hp,gender,usia,tanggal_lahir,tanggal_kunjungan,poli,dokter,diagnosa,metode_pembayaran,biaya_kunjungan,berat_badan,tinggi_badan,suhu_tubuh,status_kunjungan,rating_layanan
0,K000001,3794026542351161,Agus Kusuma,agus267@outlook.com,088120 2665686,NaN,NaN,NaN,2023-11-01,gigi,Dr. Andi,Hipertensi,BPJS Kesehatan,0.0,94.0,178.0,36.5,Batal,1.0
1,K000002,4127049386204485,Rudi Pratama,dewi496@yahoo.com,+628912573498,Female,65.0,1961-11-17,2022-05-30,Gigi,NaN,Hipertensi,cash,670321.0,89.0,152.0,36.5,Menunggu,10.0
2,K000003,123456,Agus Pratama,fajar965@gmail.com,628553946074,l,19.0,2007-11-06,2023-10-07,Mata,Dr. Andi,flu,BPJS Kesehatan,0.0,57.0,165.0,38.0,Selesai,1.0
3,K000004,1316475255341928,Maya Lestari,fajar93@icloud.com,08747296694,NaN,89.0,1937-11-23,2026-06-27,Umum,Dr. Andi,Hipertensi,cash,1820312.0,108.0,148.0,50.0,Menunggu,2.0
4,K000005,2764835030564139,Agus Kusuma,maya392@icloud.com,123456789,Male,51.0,1975-11-14,2025-12-28,Penyakit Dalam,Dr. Andi,flu,cash,1530632.0,96.0,147.0,39.0,Batal,10.0


In [ ]:
def profile_data(df, name):
    print(f"{'='*50}")
    print(f"PROFILING: {name}")
    print(f"{'='*50}")
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print(f"\nMissing Values:")
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.sum() > 0 else "None")
    print(f"\nDuplicate Rows: {df.duplicated().sum()}")
    print()

profile_data(df, 'CLINIC')

PROFILING: CLINIC
Rows: 1000
Columns: 19

Missing Values:
nik                   28
nama_pasien           20
email_pasien          43
no_hp                 50
gender               138
usia                  42
tanggal_lahir         63
tanggal_kunjungan     31
poli                 114
dokter               209
diagnosa             131
metode_pembayaran    154
biaya_kunjungan      154
berat_badan           10
tinggi_badan          10
suhu_tubuh           160
rating_layanan       146
dtype: int64

Duplicate Rows: 0



Informasi dataset:
1. Jumlah baris: 1.000
2. Jumlah kolom: 19
3. Duplicate rows: 0
4. Missing values ditemukan pada 17 kolom.

# Identifikasi Masalah Data Quality

1. Completeness (Kelengkapan)
    Menemukan missing values pada kolom dalam dataset.
2. Uniqueness (Keunikan)
    Menemukan duplicate rows pada ID kunjungan untuk memastikan setiap kunjungan memiliki ID yang unik. Memeriksa NIK yang berulang, akan tetapi tidak selalu menunjukkan duplikasi karena satu pasien dapat melakukan beberapa kunjungan.
3. Validity (Validitas)
    Memeriksa kesesuaian format dan nilai pada kolom email, NIK, dan nomor telepon untuk memastikan data memenuhi aturan yang ditentukan.
4. Consistency (Konsistensi)
    Memeriksa keseragaman penulisan nomor telepon, gender, poli, dokter, metode pembayaran, serta format tanggal lahir dan tanggal kunjungan.
5. Timeliness (Ketepatan Waktu)
    Memeriksa rentang tanggal lahir dan tanggal kunjungan untuk mengetahui periode data.

## **1. Completeness**

In [ ]:
# Hitung completeness per kolom
def check_completeness(df, name):
    total = len(df)

    result = pd.DataFrame({
        'missing': df.isnull().sum(),
        'completeness_%': (
            (1 - df.isnull().sum() / total) * 100
        ).round(2)
    })

    print(f"\n{name} Completeness:")
    print(result[result['missing'] > 0])

    return result

completeness_result = check_completeness(df, 'CLINIC')


CLINIC Completeness:
                   missing  completeness_%
nik                     28            97.2
nama_pasien             20            98.0
email_pasien            43            95.7
no_hp                   50            95.0
gender                 138            86.2
usia                    42            95.8
tanggal_lahir           63            93.7
tanggal_kunjungan       31            96.9
poli                   114            88.6
dokter                 209            79.1
diagnosa               131            86.9
metode_pembayaran      154            84.6
biaya_kunjungan        154            84.6
berat_badan             10            99.0
tinggi_badan            10            99.0
suhu_tubuh             160            84.0
rating_layanan         146            85.4


Analisis:
* Kolom berat_badan dan tinggi_badan memiliki kelengkapan tertinggi, yaitu 99%.
* Kolom dokter memiliki missing values terbanyak, yaitu 209 data, dengan completeness 79,1%. Dan kolom suhu_tubuh memiliki 160 missing values dengan completeness 84%.

## **2. Uniqueness**

In [ ]:
# Cek duplicate seluruh rows
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\nDuplicate rows:")
df[df.duplicated(keep=False)].sort_values('id_kunjungan')

Duplicate rows: 0

Duplicate rows:


,id_kunjungan,nik,nama_pasien,email_pasien,no_hp,gender,usia,tanggal_lahir,tanggal_kunjungan,poli,dokter,diagnosa,metode_pembayaran,biaya_kunjungan,berat_badan,tinggi_badan,suhu_tubuh,status_kunjungan,rating_layanan


In [ ]:
# Cek duplicate ID kunjungan
print(f"Duplicate ID Kunjungan: {df['id_kunjungan'].duplicated().sum()}")

print("\nDuplicate ID Kunjungan:")
df[df['id_kunjungan'].duplicated(keep=False)].sort_values('id_kunjungan').head()

Duplicate ID Kunjungan: 15

Duplicate ID Kunjungan:


,id_kunjungan,nik,nama_pasien,email_pasien,no_hp,gender,usia,tanggal_lahir,tanggal_kunjungan,poli,dokter,diagnosa,metode_pembayaran,biaya_kunjungan,berat_badan,tinggi_badan,suhu_tubuh,status_kunjungan,rating_layanan
25,K000026,5634216073375433,NaN,maya619@icloud.com,+628704747648,L,30.0,1996-11-08,2023-10-02,Mata,NaN,Batuk,cash,379970.0,63.0,160.0,37.0,Batal,NaN
759,K000026,6100231177533126,Fajar Wijaya,maya827@hotmail.com,628410406291,l,82.0,1944-11-21,31/12/2025,Gigi,NaN,Batuk,cash,118838.0,99.0,180.0,50.0,Batal,4.0
27,K000028,5414586850142940,Budi Lestari,siti57@outlook.com,+628071245531,P,1.0,2025-11-01,2023-10-07,gigi,Dr. Siti,NaN,BPJS,0.0,88.0,149.0,38.0,Menunggu,1.0
574,K000028,0337933756362050,Rudi Lestari,andi131@gmail.com,08627960242,L,86.0,1940-11-22,2025-06-10,umum,Dr. Siti,Diabetes,Insurance,831832.0,56.0,176.0,50.0,Menunggu,4.0
30,K000031,6629946804436995,Rina Santoso,agus142@icloud.com,628958558522,Female,75.0,1951-11-20,NaN,gigi,Dr. Budi,NaN,cash,482555.0,300.0,193.0,50.0,Menunggu,2.0


In [ ]:
# Cek duplikat NIK
print(f"Duplicate NIK: {df['nik'].duplicated().sum()}")

df[df['nik'].duplicated(keep=False)].sort_values('nik').head()

Duplicate NIK: 73


,id_kunjungan,nik,nama_pasien,email_pasien,no_hp,gender,usia,tanggal_lahir,tanggal_kunjungan,poli,dokter,diagnosa,metode_pembayaran,biaya_kunjungan,berat_badan,tinggi_badan,suhu_tubuh,status_kunjungan,rating_layanan
572,K000573,0632543107427990,Siti Santoso,siti.gmail.com,0812ABC345,l,87.0,1939-11-23,2025-09-06,Gigi,Dr. Rina,Hipertensi,Insurance,500588.0,62.0,143.0,37.0,Batal,5.0
445,K000446,0632543107427990,Rudi Santoso,putri948@yahoo.com,02149703506,NaN,13.0,2013-11-04,2023-07-30,Mata,NaN,NaN,Insurance,1290869.0,84.0,176.0,NaN,Selesai,5.0
2,K000003,123456,Agus Pratama,fajar965@gmail.com,628553946074,l,19.0,2007-11-06,2023-10-07,Mata,Dr. Andi,flu,BPJS Kesehatan,0.0,57.0,165.0,38.0,Selesai,1.0
441,K000442,123456,Budi Lestari,putri480@hotmail.com,08219051881,p,11.0,2015-11-04,2025-01-18,Umum,Dr. Budi,flu,cash,1122701.0,105.0,179.0,36.5,Selesai,4.0
49,K000050,123456,Agus Wijaya,budi311@yahoo.com,088128 1902217,Male,70.0,1956-11-18,2021-12-25,Penyakit Dalam,Dr. Andi,Hipertensi,Insurance,328473.0,54.0,157.0,36.5,Selesai,1.0


## **3. Validity**

### Email

In [ ]:
# Validasi email dengan regex
import re

def is_valid_email(email):
    if pd.isna(email):
        return False

    pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    return bool(re.match(pattern, email))


df['email_valid'] = df['email_pasien'].apply(is_valid_email)

# Email tidak valid (tanpa null)
invalid_email = df[
    ~df['email_valid'] & df['email_pasien'].notna()
]

print(f"Email tidak valid tanpa null: {len(invalid_email)}")

invalid_email[
    ['id_kunjungan', 'nama_pasien', 'email_pasien']
].head()

Email tidak valid tanpa null: 115


,id_kunjungan,nama_pasien,email_pasien
20,K000021,Rina Hidayat,maya@gmail.com
56,K000057,Fajar Kusuma,maya@gmail.com
68,K000069,Maya Lestari,siti.gmail.com
69,K000070,Budi Santoso,budi@gmail.com
96,K000097,Siti Hidayat,andi@@gmail.com


In [ ]:
invalid_email = df[~df['email_valid']]

print(f"Email tidak valid (termasuk null): {len(invalid_email)}")

invalid_email.head()

Email tidak valid (termasuk null): 158


,id_kunjungan,nik,nama_pasien,email_pasien,no_hp,gender,usia,tanggal_lahir,tanggal_kunjungan,poli,dokter,diagnosa,metode_pembayaran,biaya_kunjungan,berat_badan,tinggi_badan,suhu_tubuh,status_kunjungan,rating_layanan,email_valid
20,K000021,7849808412411824,Rina Hidayat,maya@gmail.com,0812ABC345,p,4.0,2022-11-02,2025-05-07,Penyakit Dalam,Dr. Siti,Demam,BPJS,0.0,108.0,187.0,39.0,Batal,NaN,False
46,K000047,7770143634957885,Maya Santoso,NaN,+628211896844,NaN,75.0,1951-11-20,2024-09-19,Penyakit Dalam,Dr. Siti,Diabetes,Cash,478192.0,79.0,198.0,38.0,Selesai,3.0,False
56,K000057,1399049027874296,Fajar Kusuma,maya@gmail.com,628177905528,P,34.0,1992-11-09,2022-12-19,Anak,Dr. Rina,Batuk,Insurance,129725.0,99.0,199.0,50.0,Menunggu,2.0,False
60,K000061,1317127484677378,Maya Pratama,NaN,628609085818,p,60.0,1966-11-16,2022-02-19,Penyakit Dalam,Dr. Andi,Demam,BPJS,50000.0,58.0,181.0,36.5,Selesai,5.0,False
68,K000069,4740748217594647,Maya Lestari,siti.gmail.com,628896660534,Female,-5.0,2031-10-31,2024-09-28,Gigi,Dr. Budi,Batuk,cash,1858811.0,89.0,195.0,37.0,Batal,2.0,False


Ditemukan email yang seharusnya valid tetapi hasilnya False

In [ ]:
repr(df.loc[20, 'email_pasien'])

"'maya@gmail.com '"

Kesimpulan validasi email:

Berdasarkan hasil validasi menggunakan regex, ditemukan beberapa email yang tidak sesuai dengan format yang ditentukan, seperti email tanpa tanda @, domain yang tidak lengkap, penggunaan tanda @ lebih dari satu, serta email dengan karakter tambahan. Dan juga terdapat email spasi lebih.
Data email yang tidak valid akan diperbaiki pada tahap Cleaning

### NIK

In [ ]:
# Validasi NIK harus 16 digit angka
df['nik_valid'] = (
    df['nik'].astype('string')
    .str.fullmatch(r'\d{16}')
    .fillna(False)
)

invalid_nik = df[
    ~df['nik_valid'] & df['nik'].notna()
]

print(f"NIK tidak valid tanpa null: {len(invalid_nik)}")

invalid_nik.head()

NIK tidak valid tanpa null: 30


,id_kunjungan,nik,nama_pasien,email_pasien,no_hp,gender,usia,tanggal_lahir,tanggal_kunjungan,poli,...,diagnosa,metode_pembayaran,biaya_kunjungan,berat_badan,tinggi_badan,suhu_tubuh,status_kunjungan,rating_layanan,email_valid,nik_valid
2,K000003,123456,Agus Pratama,fajar965@gmail.com,628553946074,l,19.0,2007-11-06,2023-10-07,Mata,...,flu,BPJS Kesehatan,0.0,57.0,165.0,38.0,Selesai,1.0,True,False
23,K000024,ABC123456789,Maya Wijaya,dewi887@outlook.com,08622155996,Female,200.0,NaN,2024-11-30,Mata,...,NaN,Cash,1351752.0,108.0,NaN,37.0,Batal,10.0,True,False
49,K000050,123456,Agus Wijaya,budi311@yahoo.com,088128 1902217,Male,70.0,1956-11-18,2021-12-25,Penyakit Dalam,...,Hipertensi,Insurance,328473.0,54.0,157.0,36.5,Selesai,1.0,True,False
63,K000064,123456789012345678,Budi Wijaya,rina948@icloud.com,02136627135,L,15.0,2011-11-05,2025-10-07,Penyakit Dalam,...,Flu,BPJS Kesehatan,0.0,65.0,300.0,39.0,Menunggu,10.0,True,False
71,K000072,ABC123456789,Fajar Hidayat,rudi641@icloud.com,08038802730,NaN,69.0,1957-11-18,2026-09-26,Gigi,...,Flu,cash,492974.0,63.0,152.0,NaN,Batal,3.0,True,False


In [ ]:
invalid_nik = df[~df['nik_valid']]

print(f"NIK tidak valid (termasuk null): {len(invalid_nik)}")

invalid_nik.head()

NIK tidak valid (termasuk null): 58


,id_kunjungan,nik,nama_pasien,email_pasien,no_hp,gender,usia,tanggal_lahir,tanggal_kunjungan,poli,...,diagnosa,metode_pembayaran,biaya_kunjungan,berat_badan,tinggi_badan,suhu_tubuh,status_kunjungan,rating_layanan,email_valid,nik_valid
2,K000003,123456,Agus Pratama,fajar965@gmail.com,628553946074,l,19.0,2007-11-06,2023-10-07,Mata,...,flu,BPJS Kesehatan,0.0,57.0,165.0,38.0,Selesai,1.0,True,False
11,K000012,NaN,Rina Lestari,siti159@icloud.com,08026270664,Male,10.0,2016-11-03,2024-07-03,Umum,...,flu,BPJS Kesehatan,50000.0,55.0,147.0,39.0,Selesai,4.0,True,False
23,K000024,ABC123456789,Maya Wijaya,dewi887@outlook.com,08622155996,Female,200.0,NaN,2024-11-30,Mata,...,NaN,Cash,1351752.0,108.0,NaN,37.0,Batal,10.0,True,False
26,K000027,NaN,Putri Santoso,siti585@yahoo.com,+628593965213,Female,73.0,1953-11-19,2025-06-06,Mata,...,Demam,BPJS,0.0,80.0,151.0,37.0,Batal,10.0,True,False
32,K000433,NaN,Rina Wijaya,BUDI@GMAIL.COM,08710508054,L,14.0,2012-11-04,2025-01-19,gigi,...,Hipertensi,cash,1545874.0,53.0,168.0,37.0,Selesai,10.0,True,False


Kesimpulan validasi NIK:

Ditemukan NIK tidak memenuhi format 16 digit angka. Beberapa NIK mengandung huruf, memiliki jumlah digit kurang dari 16, dan memiliki jumlah digit lebih dari 16. Data tersebut perlu diperbaiki

### No.HP

In [25]:
# Validasi nomor telepon Indonesia
phone = (
    df['no_hp']
    .astype('string')
    .str.replace(r'[\s-]', '', regex=True)
)

df['no_hp_valid'] = (
    phone.str.fullmatch(r'\+?\d{11,13}')
    .fillna(False)
)

invalid_phone = df[
    ~df['no_hp_valid'] & df['no_hp'].notna()
]

print(f"Nomor telepon tidak valid: {len(invalid_phone)}")

invalid_phone[['id_kunjungan', 'nama_pasien', 'no_hp']].head()

Nomor telepon tidak valid: 66


,id_kunjungan,nama_pasien,no_hp
4,K000005,Agus Kusuma,123456789
7,K000008,Putri Lestari,628123456789012345
20,K000021,Rina Hidayat,0812ABC345
53,K000054,Andi Lestari,0812ABC345
62,K000063,Rudi Santoso,+62-ABC


In [26]:
invalid_phone = df[~df['no_hp_valid']]

print(f"No hp tidak valid (termasuk null): {len(invalid_phone)}")

invalid_phone.head()

No hp tidak valid (termasuk null): 116


,id_kunjungan,nik,nama_pasien,email_pasien,no_hp,gender,usia,tanggal_lahir,tanggal_kunjungan,poli,...,metode_pembayaran,biaya_kunjungan,berat_badan,tinggi_badan,suhu_tubuh,status_kunjungan,rating_layanan,email_valid,nik_valid,no_hp_valid
4,K000005,2764835030564139,Agus Kusuma,maya392@icloud.com,123456789,Male,51.0,1975-11-14,2025-12-28,Penyakit Dalam,...,cash,1530632.0,96.0,147.0,39.0,Batal,10.0,True,True,False
7,K000008,8018451462704828,Putri Lestari,maya180@gmail.com,628123456789012345,NaN,-5.0,2031-10-31,2026-09-10,Gigi,...,cash,684287.0,85.0,177.0,36.5,Selesai,5.0,True,True,False
15,K000016,8013267736026064,Putri Lestari,putri148@yahoo.com,NaN,Male,1.0,2025-11-01,2024-01-31,Penyakit Dalam,...,cash,1076487.0,300.0,148.0,38.0,Menunggu,5.0,True,True,False
20,K000021,7849808412411824,Rina Hidayat,maya@gmail.com,0812ABC345,p,4.0,2022-11-02,2025-05-07,Penyakit Dalam,...,BPJS,0.0,108.0,187.0,39.0,Batal,NaN,False,True,False
45,K000046,2409036788127868,Andi Kusuma,fajar921@gmail.com,NaN,Female,60.0,1966-11-16,2022-11-14,Mata,...,cash,1343398.0,55.0,196.0,37.0,Selesai,NaN,True,True,False


Berdasarkan hasil validasi, ditemukan beberapa nomor telepon yang tidak memenuhi format yang ditentukan, seperti nomor dengan jumlah digit yang tidak sesuai dari 11-13. Perbedaan format penulisan tersebut akan diketahui pada tahap Consistency.

## **4. Consistency**

In [27]:
# Cek konsistensi kategori
print("Gender unik:")
print(df['gender'].unique())

print("\nMetode pembayaran unik:")
print(df['metode_pembayaran'].unique())

print("\nPoli unik:")
print(df['poli'].unique())

print("\nDokter unik:")
print(df['dokter'].unique())

print("\nStatus kunjungan unik:")
print(df['status_kunjungan'].unique())

Gender unik:
[nan 'Female' 'l' 'Male' 'P' 'p' 'L']

Metode pembayaran unik:
['BPJS Kesehatan' 'cash' 'Cash' 'Insurance' nan 'BPJS']

Poli unik:
['gigi' 'Gigi' 'Mata' 'Umum' 'Penyakit Dalam' 'Anak' nan 'umum']

Dokter unik:
['Dr. Andi' nan 'Dr. Rina' 'Dr. Siti' 'Dr. Budi']

Status kunjungan unik:
['Batal' 'Menunggu' 'Selesai']


In [28]:
# Cek variasi no. Hp
print("Variasi awalan nomor HP:")
print(
    df['no_hp']
    .dropna()
    .str.strip()
    .str.extract(r'^(\+?62|0\d{1,3})')[0]
    .value_counts()
)

Variasi awalan nomor HP:
0
62      216
+62     199
0881     88
0812     28
0214     24
       ... 
0870      1
0895      1
0894      1
0839      1
0817      1
Name: count, Length: 104, dtype: int64


In [30]:
# Cek variasi format tanggal kunjungan
print("Tanggal kunjungan unik:")# Cek konsistensi format tanggal kunjungan
visit_date  = df['tanggal_kunjungan'].dropna().astype(str)

format_ymd = visit_date.str.match(r'^\d{4}-\d{2}-\d{2}$')
format_dmy = visit_date.str.match(r'^\d{2}/\d{2}/\d{4}$')

print("Format YYYY-MM-DD:", format_ymd.sum())
print("Format DD/MM/YYYY:", format_dmy.sum())
print("Format lainnya:", (~(format_ymd | format_dmy)).sum())

Tanggal kunjungan unik:
Format YYYY-MM-DD: 883
Format DD/MM/YYYY: 86
Format lainnya: 0


In [31]:
# Cek konsistensi format tanggal lahir
birth_date = df['tanggal_lahir'].dropna().astype(str)

format_ymd = birth_date.str.match(r'^\d{4}-\d{2}-\d{2}$')
format_dmy = birth_date.str.match(r'^\d{2}/\d{2}/\d{4}$')

print("Format YYYY-MM-DD:", format_ymd.sum())
print("Format DD/MM/YYYY:", format_dmy.sum())
print("Format lainnya:", (~(format_ymd | format_dmy)).sum())

Format YYYY-MM-DD: 937
Format DD/MM/YYYY: 0
Format lainnya: 0


## **5. Timeliness**

In [32]:
# Cek data range tanggal lahir dan tanggal kunjungan

#Convert ke datetime
df['tanggal_lahir'] = pd.to_datetime(
    df['tanggal_lahir'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

df['tanggal_kunjungan'] = pd.to_datetime(
    df['tanggal_kunjungan'].astype('string').str.strip(),
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

print(
    f"   - Tanggal kunjungan: "
    f"{df['tanggal_kunjungan'].min()} sampai "
    f"{df['tanggal_kunjungan'].max()}"
)

print(
    f"   - Tanggal lahir: "
    f"{df['tanggal_lahir'].min()} sampai "
    f"{df['tanggal_lahir'].max()}"
)

   - Tanggal kunjungan: 2021-05-11 00:00:00 sampai 2030-01-01 00:00:00
   - Tanggal lahir: 1936-11-23 00:00:00 sampai 2031-10-31 00:00:00


In [33]:
# Date Format
print("Tanggal lahir:", df['tanggal_lahir'].dtype)
print("Tanggal kunjungan:", df['tanggal_kunjungan'].dtype)

print(df[['tanggal_lahir', 'tanggal_kunjungan']].head(10))

Tanggal lahir: datetime64[ns]
Tanggal kunjungan: datetime64[ns]
  tanggal_lahir tanggal_kunjungan
0           NaT        2023-11-01
1    1961-11-17        2022-05-30
2    2007-11-06        2023-10-07
3    1937-11-23        2026-06-27
4    1975-11-14        2025-12-28
5    1943-11-22        2023-02-18
6           NaT        2022-12-12
7    2031-10-31        2026-09-10
8    1983-11-12        2025-11-24
9    1949-11-20        2023-08-08


In [38]:
# Identifikasi pasien unik tidak berkunjung >180 hari

today = pd.to_datetime('2026-11-08')

nik_valid = df['nik'].astype('string').str.fullmatch(r'\d{16}')

last_visit = df.groupby('nik')['tanggal_kunjungan'].transform('max')

df['days_since_visit'] = (today - last_visit).dt.days
df['is_inactive'] = (df['days_since_visit'] > 180) & nik_valid.fillna(False)

# Daftar pasien tidak aktif (satu baris per NIK)
inactive_patients = df[df['is_inactive']].drop_duplicates('nik')

print(f"Total pasien tidak berkunjung >180 hari: {len(inactive_patients)}")

print("\nDaftar pasien tidak aktif:")
inactive_patients[
    ['nik', 'nama_pasien', 'tanggal_kunjungan', 'days_since_visit']
]

Total pasien tidak berkunjung >180 hari: 807

Daftar pasien tidak aktif:


,nik,nama_pasien,tanggal_kunjungan,days_since_visit
0,3794026542351161,Agus Kusuma,2023-11-01,1103.0
4,2764835030564139,Agus Kusuma,2025-12-28,315.0
5,5376724238849696,Dewi Wijaya,2023-02-18,1359.0
6,2871012269166978,Agus Lestari,2022-12-12,1427.0
8,4893252880957015,Andi Pratama,2025-11-24,349.0
...,...,...,...,...
994,3943804032537657,Rudi Pratama,2022-12-25,1414.0
995,9277894319100900,Rina Lestari,2024-02-28,984.0
996,0476921053365146,Agus Santoso,2023-08-19,1177.0
997,4014247310720113,Maya Kusuma,2023-10-22,1113.0


# **Data Quality Report**

In [39]:
def generate_dq_report(df, name):
    total = len(df)
    total_cells = df.size
    missing = df.isnull().sum().sum()
    duplicates = df.duplicated().sum()

    completeness = (1 - missing / total_cells) * 100
    uniqueness = (1 - duplicates / total) * 100

    print("=" * 50)
    print(f"DATA QUALITY REPORT: {name}")
    print("=" * 50)

    print(f"Total Records    : {total}")
    print(f"Total Columns    : {df.shape[1]}")
    print(f"Total Cells      : {total_cells}")
    print(f"Missing Values   : {missing}")
    print(f"Duplicate Rows   : {duplicates}")

    print("\nDQ Scores:")
    print(f"  Completeness   : {completeness:.2f}%")
    print(f"  Uniqueness     : {uniqueness:.2f}%")

    print("=" * 50)


generate_dq_report(df, "DATASET KLINIK")

DATA QUALITY REPORT: DATASET KLINIK
Total Records    : 1000
Total Columns    : 24
Total Cells      : 24000
Missing Values   : 1557
Duplicate Rows   : 0

DQ Scores:
  Completeness   : 93.51%
  Uniqueness     : 100.00%


In [41]:
# Summary semua issues
print("="*60)
print("SUMMARY: DATA QUALITY ISSUES FOUND")
print("="*60)

print("\n1. COMPLETENESS (Missing Data)")
print("   - Ditemukan missing values pada 17 kolom")

print("\n2. UNIQUENESS (Duplicates)")
print(f"   - Duplicate id_kunjungan: {df['id_kunjungan'].duplicated().sum()}")
print(f"   - Duplicate NIK: {df['nik'].dropna().duplicated().sum()}")

print("\n3. VALIDITY (Format/Value Errors)")
print(f"   - Email format tidak valid: 115")
print(f"   - NIK tidak valid: 30")
print(f"   - Nomor telepon tidak valid: 66")

print("\n4. CONSISTENCY (Inconsistent Values)")
print("   - Gender: perlu diperiksa keseragaman penulisan")
print("   - Nomor telepon: ditemukan variasi awalan 08, 62, +62, dan 021")
print("   - Poli: perlu diperiksa keseragaman penulisan")
print("   - Metode pembayaran: perlu diperiksa keseragaman penulisan")
print("   - Tanggal kunjungan: ditemukan dua format, YYYY-MM-DD dan DD/MM/YYYY")

print("\n5. TIMELINESS (Outdated Data)")
print(f"   - Pasien tidak berkunjung >180 hari: 807")

print("="*60)

SUMMARY: DATA QUALITY ISSUES FOUND

1. COMPLETENESS (Missing Data)
   - Ditemukan missing values pada 17 kolom

2. UNIQUENESS (Duplicates)
   - Duplicate id_kunjungan: 15
   - Duplicate NIK: 46

3. VALIDITY (Format/Value Errors)
   - Email format tidak valid: 115
   - NIK tidak valid: 30
   - Nomor telepon tidak valid: 66

4. CONSISTENCY (Inconsistent Values)
   - Gender: perlu diperiksa keseragaman penulisan
   - Nomor telepon: ditemukan variasi awalan 08, 62, +62, dan 021
   - Poli: perlu diperiksa keseragaman penulisan
   - Metode pembayaran: perlu diperiksa keseragaman penulisan
   - Tanggal kunjungan: ditemukan dua format, YYYY-MM-DD dan DD/MM/YYYY

5. TIMELINESS (Outdated Data)
   - Pasien tidak berkunjung >180 hari: 807


# **Correcting Issues**

In [60]:
# Copy dataset
df_clean = df.copy()


# 1. CONSISTENCY
df_clean['gender'] = df_clean['gender'].str.strip().str.upper().replace({
    'FEMALE': 'P', 'MALE': 'L'
})

df_clean['metode_pembayaran'] = df_clean['metode_pembayaran'].str.strip().str.upper().replace({
    'BPJS KESEHATAN': 'BPJS'
})

for col in ['poli', 'dokter', 'status_kunjungan']:
    df_clean[col] = df_clean[col].str.strip().str.upper()


# 2. COMPLETENESS
numeric_cols = [
    'usia', 'biaya_kunjungan', 'berat_badan',
    'tinggi_badan', 'suhu_tubuh'
]

categorical_cols = [
    'gender', 'poli', 'dokter', 'metode_pembayaran'
]

other_cols = [
    'nama_pasien', 'nik', 'no_hp',
    'email_pasien', 'diagnosa'
]

for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

for col in other_cols:
    df_clean[col] = df_clean[col].fillna('Tidak diketahui')


# 3. UNIQUENESS
df_clean = df_clean.drop_duplicates(subset=['id_kunjungan'])


# 4. VALIDITY
for col in ['email_pasien', 'nik', 'no_hp']:
    df_clean[col] = df_clean[col].str.strip()

invalid_email = ~email_valid.loc[df_clean.index]
df_clean.loc[invalid_email[invalid_email].index, 'email_pasien'] = 'Tidak diketahui'

invalid_nik = ~nik_valid.loc[df_clean.index]
df_clean.loc[invalid_nik[invalid_nik].index, 'nik'] = 'Tidak diketahui'

invalid_phone = ~no_hp_valid.loc[df_clean.index]
df_clean.loc[invalid_phone[invalid_phone].index, 'no_hp'] = 'Tidak diketahui'


# 5. STANDARDISASI NOMOR HP
phone_mask = df_clean['no_hp'].notna() & (df_clean['no_hp'] != 'Tidak diketahui')

df_clean.loc[phone_mask, 'no_hp'] = (
    df_clean.loc[phone_mask, 'no_hp']
    .str.replace(r'[\s-]', '', regex=True)
    .str.replace(r'^\+?62', '0', regex=True)
)


# HASIL CLEANING
print("Cleaning selesai!")
print(f"Jumlah data awal: {len(df)}")
print(f"Jumlah data setelah cleaning: {len(df_clean)}")
print(f"Jumlah data duplikat yang dihapus: {len(df) - len(df_clean)}")

Cleaning selesai!
Jumlah data awal: 1000
Jumlah data setelah cleaning: 985
Jumlah data duplikat yang dihapus: 15


Proses cleaning berhasil menghilangkan duplikasi ID kunjungan, menangani nilai email, NIK, dan nomor telepon yang tidak valid, serta menyeragamkan penulisan kategori gender, metode pembayaran, poli, dan status kunjungan. Masih terdapat missing values pada beberapa kolom yang tidak dapat diisi tanpa informasi pendukung, khususnya data tanggal.

In [62]:
# Verify Cleaning
print("After Cleaning:")

# 1. Completeness
print("\n--- Completeness ---")
print("Missing values:", df_clean[df.columns].isnull().sum().sum())

# 2. Uniqueness
print("\n--- Uniqueness ---")
print("Duplicate rows:", df_clean.duplicated().sum())
print("Duplicate ID:", df_clean['id_kunjungan'].duplicated().sum())

# 3. Validity
print("\n--- Validity ---")

# Email
email_check = df_clean['email_pasien'].str.fullmatch(
    r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
)

print("\nEmail:")
print("Valid:", email_check.sum())
print("Invalid:", ((~email_check) & (df_clean['email_pasien'] != 'Tidak diketahui')).sum())
print("Tidak diketahui:", (df_clean['email_pasien'] == 'Tidak diketahui').sum())

# NIK
nik_check = df_clean['nik'].str.fullmatch(r'\d{16}')

print("\nNIK:")
print("Valid:", nik_check.sum())
print("Invalid:", ((~nik_check) & (df_clean['nik'] != 'Tidak diketahui')).sum())
print("Tidak diketahui:", (df_clean['nik'] == 'Tidak diketahui').sum())

# No HP
phone_check = df_clean['no_hp'].str.fullmatch(
    r'(08\d{8,11}|021\d{7,10})'
)

print("\nNo HP:")
print("Valid:", phone_check.sum())
print("Invalid:", ((~phone_check) & (df_clean['no_hp'] != 'Tidak diketahui')).sum())
print("Tidak diketahui:", (df_clean['no_hp'] == 'Tidak diketahui').sum())

# 4. Consistency
print("\n--- Consistency ---")
print("Gender:", df_clean['gender'].unique())
print("Metode pembayaran:", df_clean['metode_pembayaran'].unique())
print("Poli:", df_clean['poli'].unique())

# 5. Date Format
print("\n--- Date Format ---")
print("Tanggal kunjungan:", df_clean['tanggal_kunjungan'].dtype)

After Cleaning:

--- Completeness ---
Missing values: 290

--- Uniqueness ---
Duplicate rows: 0
Duplicate ID: 0

--- Validity ---

Email:
Valid: 862
Invalid: 0
Tidak diketahui: 123

NIK:
Valid: 929
Invalid: 0
Tidak diketahui: 56

No HP:
Valid: 873
Invalid: 0
Tidak diketahui: 112

--- Consistency ---
Gender: ['L' 'P']
Metode pembayaran: ['BPJS' 'CASH' 'INSURANCE']
Poli: ['GIGI' 'MATA' 'UMUM' 'PENYAKIT DALAM' 'ANAK']

--- Date Format ---
Tanggal kunjungan: datetime64[ns]


In [63]:
# Cek missing values setelah cleaning
print("Missing Values After Cleaning:")
print(df_clean[df.columns].isnull().sum())

# Total missing values
print("\nTotal missing values:", df_clean[df.columns].isnull().sum().sum())

Missing Values After Cleaning:
id_kunjungan           0
nik                    0
nama_pasien            0
email_pasien           0
no_hp                  0
gender                 0
usia                   0
tanggal_lahir         63
tanggal_kunjungan     30
poli                   0
dokter                 0
diagnosa               0
metode_pembayaran      0
biaya_kunjungan        0
berat_badan            0
tinggi_badan           0
suhu_tubuh             0
status_kunjungan       0
rating_layanan       145
email_valid            0
nik_valid              0
no_hp_valid            0
days_since_visit      52
is_inactive            0
dtype: int64

Total missing values: 290


Proses cleaning pada aspek completeness berhasil mengatasi missing values pada sebagian besar kolom. Sebanyak 290 missing values masih dipertahankan karena memerlukan penanganan khusus atau informasi aslinya memang tidak tersedia.